[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-3/streaming-interruption.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/58239464-lesson-1-streaming)

# 流式传输（Streaming）

## 回顾

在模块 2 中，我们介绍了几种自定义图状态和记忆的方法。

我们构建了一个带有外部记忆的聊天机器人，可以维持长期对话。

## 目标

本模块将深入探讨"人机协同"（`human-in-the-loop`），它建立在记忆之上，允许用户以各种方式直接与图进行交互。

为了为"人机协同"做准备，我们将首先深入探讨流式传输，它提供了多种在执行过程中可视化图输出（例如节点状态或聊天模型 token）的方法。

In [ ]:
%%capture --no-stderr
%pip install --quiet -U langgraph langchain_deepseek langgraph_sdk

## 流式传输

LangGraph 构建时提供了[对流式传输的一流支持](https://docs.langchain.com/oss/python/langgraph/streaming)。

让我们设置模块 2 中的聊天机器人，并展示在执行过程中流式输出图结果的各种方式。

In [ ]:
import os, getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("DEEPSEEK_API_KEY")

请注意，我们在 `call_model` 中使用了 `RunnableConfig` 来启用逐 token 流式传输。这[仅在使用 Python < 3.11 时需要](https://langchain-ai.github.io/langgraph/how-tos/streaming-tokens/)。我们包含它是为了以防你在 Colab 中运行此 notebook（将使用 Python 3.x）。

In [ ]:
from IPython.display import Image, display
from typing import Literal

from langchain_deepseek import ChatDeepSeek
from langchain_core.messages import SystemMessage, HumanMessage, RemoveMessage
from langchain_core.runnables import RunnableConfig

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, START, END
from langgraph.graph import MessagesState

# LLM
model = ChatDeepSeek(model="deepseek-chat", temperature=0) 

# 状态
class State(MessagesState):
    summary: str

# 定义调用模型的逻辑
def call_model(state: State, config: RunnableConfig):
    
    # 获取摘要（如果存在）
    summary = state.get("summary", "")

    # 如果有摘要，则添加它
    if summary:
        
        # 将摘要添加到系统消息
        system_message = f"Summary of conversation earlier: {summary}"

        # 将摘要追加到任何较新的消息之前
        messages = [SystemMessage(content=system_message)] + state["messages"]
    
    else:
        messages = state["messages"]
    
    response = model.invoke(messages, config)
    return {"messages": response}

def summarize_conversation(state: State):
    
    # 首先，获取任何现有的摘要
    summary = state.get("summary", "")

    # 创建我们的摘要提示
    if summary:
        
        # 摘要已存在
        summary_message = (
            f"This is summary of the conversation to date: {summary}\n\n"
            "Extend the summary by taking into account the new messages above:"
        )
        
    else:
        summary_message = "Create a summary of the conversation above:"

    # 将提示添加到我们的历史记录中
    messages = state["messages"] + [HumanMessage(content=summary_message)]
    response = model.invoke(messages)
    
    # 删除除最近 2 条消息以外的所有消息
    delete_messages = [RemoveMessage(id=m.id) for m in state["messages"][:-2]]
    return {"summary": response.content, "messages": delete_messages}

# 判断是终止还是总结对话
def should_continue(state: State)-> Literal ["summarize_conversation",END]:
    
    """返回下一个要执行的节点。"""
    
    messages = state["messages"]
    
    # 如果消息超过 6 条，则总结对话
    if len(messages) > 6:
        return "summarize_conversation"
    
    # 否则直接终止
    return END

# 定义一个新图
workflow = StateGraph(State)
workflow.add_node("conversation", call_model)
workflow.add_node(summarize_conversation)

# 设置入口点为 conversation
workflow.add_edge(START, "conversation")
workflow.add_conditional_edges("conversation", should_continue)
workflow.add_edge("summarize_conversation", END)

# 编译
memory = MemorySaver()
graph = workflow.compile(checkpointer=memory)
display(Image(graph.get_graph().draw_mermaid_png()))

### 流式传输完整状态

现在，让我们讨论[流式传输图状态](https://docs.langchain.com/oss/python/langgraph/streaming#supported-stream-modes)的方式。

`.stream` 和 `.astream` 是用于流式返回结果的同步和异步方法。

LangGraph 支持几种不同的图状态[流式传输模式](https://docs.langchain.com/oss/python/langgraph/streaming#stream-graph-state)：

* `values`：在每个节点被调用后流式传输图的完整状态。
* `updates`：在每个节点被调用后流式传输图状态的更新。

![values_vs_updates.png](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66dbaf892d24625a201744e5_streaming1.png)

让我们看一下 `stream_mode="updates"`。

因为我们使用 `updates` 进行流式传输，所以只能看到图中每个节点运行后状态的更新。

每个 `chunk` 是一个字典，`node_name` 为键，更新后的状态为值。

In [ ]:
# 创建线程
config = {"configurable": {"thread_id": "1"}}

# 开始对话
for chunk in graph.stream({"messages": [HumanMessage(content="hi! I'm Lance")]}, config, stream_mode="updates"):
    print(chunk)

现在让我们只打印状态更新。

In [ ]:
# 开始对话
for chunk in graph.stream({"messages": [HumanMessage(content="hi! I'm Lance")]}, config, stream_mode="updates"):
    chunk['conversation']["messages"].pretty_print()

现在，我们可以查看 `stream_mode="values"`。

这是 `conversation` 节点被调用后图的`完整状态`。

In [ ]:
# 再次开始对话
config = {"configurable": {"thread_id": "2"}}

# 开始对话
input_message = HumanMessage(content="hi! I'm Lance")
for event in graph.stream({"messages": [input_message]}, config, stream_mode="values"):
    for m in event['messages']:
        m.pretty_print()
    print("---"*25)

### 流式传输 Token

我们通常希望传输的不仅仅是图状态。

特别是在聊天模型调用中，常见的是在生成 token 时逐个流式传输它们。

我们可以[使用 `.astream_events` 方法](https://docs.langchain.com/oss/python/langchain/models#advanced-streaming-topics:streaming-events)来实现，它会在节点内部事件发生时流式返回这些事件！

每个事件是一个包含几个键的字典：

* `event`：正在发出的事件类型。
* `name`：事件的名称。
* `data`：与事件关联的数据。
* `metadata`：包含 `langgraph_node`，即发出事件的节点。

让我们看一下。

In [ ]:
config = {"configurable": {"thread_id": "3"}}
input_message = HumanMessage(content="Tell me about the 49ers NFL team")
async for event in graph.astream_events({"messages": [input_message]}, config, version="v2"):
    print(f"Node: {event['metadata'].get('langgraph_node','')}. Type: {event['event']}. Name: {event['name']}")

核心点是图中聊天模型产生的 token 具有 `on_chat_model_stream` 类型。

我们可以使用 `event['metadata']['langgraph_node']` 来选择要从哪个节点流式传输。

我们可以使用 `event['data']` 来获取每个事件的实际数据，在本例中是一个 `AIMessageChunk`。

In [ ]:
node_to_stream = 'conversation'
config = {"configurable": {"thread_id": "4"}}
input_message = HumanMessage(content="Tell me about the 49ers NFL team")
async for event in graph.astream_events({"messages": [input_message]}, config, version="v2"):
    # 获取来自特定节点的聊天模型 token
    if event["event"] == "on_chat_model_stream" and event['metadata'].get('langgraph_node','') == node_to_stream:
        print(event["data"])

如上所示，只需使用 `chunk` 键来获取 `AIMessageChunk`。

In [ ]:
config = {"configurable": {"thread_id": "5"}}
input_message = HumanMessage(content="Tell me about the 49ers NFL team")
async for event in graph.astream_events({"messages": [input_message]}, config, version="v2"):
    # 获取来自特定节点的聊天模型 token
    if event["event"] == "on_chat_model_stream" and event['metadata'].get('langgraph_node','') == node_to_stream:
        data = event["data"]
        print(data["chunk"].content, end="|")

### 通过 LangGraph API 进行流式传输

**⚠️ 注意**

自录制这些视频以来，我们已更新了 Studio，现在可以在本地运行并通过浏览器访问。这是运行 Studio 的首选方式，而非视频中展示的桌面应用。它现在名为 _LangSmith Studio_（而非 _LangGraph Studio_）。详细设置说明请参见课程开头的"环境搭建"指南。你可以在[此处](https://docs.langchain.com/langsmith/studio)找到 Studio 的说明，在[此处](https://docs.langchain.com/langsmith/quick-start-studio#local-development-server)找到本地部署的具体细节。  
要启动本地开发服务器，请在本模块的 `/studio` 目录下的终端中运行以下命令：

```
langgraph dev
```

你应该看到以下输出：
```
- 🚀 API: http://127.0.0.1:2024
- 🎨 Studio UI: https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024
- 📚 API Docs: http://127.0.0.1:2024/docs
```

打开浏览器并导航到上面显示的 **Studio UI** URL。

LangGraph API [支持编辑图状态](https://docs.langchain.com/langsmith/add-human-in-the-loop)。

In [ ]:
if 'google.colab' in str(get_ipython()):
    raise Exception("Unfortunately LangGraph Studio is currently not supported on Google Colab")

In [ ]:
from langgraph_sdk import get_client

# 这是本地开发服务器的 URL
URL = "http://127.0.0.1:2024"
client = get_client(url=URL)

# 搜索所有托管的图
assistants = await client.assistants.search()

让我们像之前一样[流式传输 `values`](https://docs.langchain.com/oss/python/langgraph/streaming#stream-graph-state)。

In [ ]:
# Create a new thread
thread = await client.threads.create()
# Input message
input_message = HumanMessage(content="Multiply 2 and 3")
async for event in client.runs.stream(thread["thread_id"], 
                                      assistant_id="agent", 
                                      input={"messages": [input_message]}, 
                                      stream_mode="values"):
    print(event)

流式传输的对象包含：

* `event`：类型
* `data`：状态

In [ ]:
from langchain_core.messages import convert_to_messages
thread = await client.threads.create()
input_message = HumanMessage(content="Multiply 2 and 3")
async for event in client.runs.stream(thread["thread_id"], assistant_id="agent", input={"messages": [input_message]}, stream_mode="values"):
    messages = event.data.get('messages',None)
    if messages:
        print(convert_to_messages(messages)[-1])
    print('='*25)

有一些新的流式传输模式仅通过 API 支持。

例如，我们可以[使用 `messages` 模式](https://docs.langchain.com/oss/python/langgraph/streaming#supported-stream-modes)更好地处理上述情况！

该模式当前假设你的图中有一个 `messages` 键，它是一个消息列表。

使用 `messages` 模式发出的所有事件都有两个属性：

* `event`：事件名称
* `data`：与事件关联的数据

In [ ]:
thread = await client.threads.create()
input_message = HumanMessage(content="Multiply 2 and 3")
async for event in client.runs.stream(thread["thread_id"], 
                                      assistant_id="agent", 
                                      input={"messages": [input_message]}, 
                                      stream_mode="messages"):
    print(event.event)

我们可以看到一些事件：

* `metadata`：有关运行的元数据
* `messages/complete`：完整的消息
* `messages/partial`：聊天模型 token

<!--你可以在此处进一步了解各种类型 [~here~](https://docs.langchain.com/oss/python/langgraph/concepts/langgraph_server)。-->

现在，让我们展示如何流式传输这些消息。

我们将定义一个辅助函数，以便更好地格式化消息中的工具调用。

In [ ]:
thread = await client.threads.create()
input_message = HumanMessage(content="Multiply 2 and 3")

def format_tool_calls(tool_calls):
    """
    将工具调用列表格式化为可读的字符串。

    Args:
        tool_calls (list): 表示工具调用的字典列表。
            每个字典应包含 'id'、'name' 和 'args' 键。

    Returns:
        str: 格式化后的工具调用字符串，如果列表为空则返回 "No tool calls"。

    """

    if tool_calls:
        formatted_calls = []
        for call in tool_calls:
            formatted_calls.append(
                f"Tool Call ID: {call['id']}, Function: {call['name']}, Arguments: {call['args']}"
            )
        return "\n".join(formatted_calls)
    return "No tool calls"

async for event in client.runs.stream(
    thread["thread_id"],
    assistant_id="agent",
    input={"messages": [input_message]},
    stream_mode="messages",):
    
    # 处理元数据事件
    if event.event == "metadata":
        print(f"Metadata: Run ID - {event.data['run_id']}")
        print("-" * 50)
    
    # 处理部分消息事件
    elif event.event == "messages/partial":
        for data_item in event.data:
            # 处理用户消息
            if "role" in data_item and data_item["role"] == "user":
                print(f"Human: {data_item['content']}")
            else:
                # 从事件中提取相关数据
                tool_calls = data_item.get("tool_calls", [])
                invalid_tool_calls = data_item.get("invalid_tool_calls", [])
                content = data_item.get("content", "")
                response_metadata = data_item.get("response_metadata", {})

                if content:
                    print(f"AI: {content}")

                if tool_calls:
                    print("Tool Calls:")
                    print(format_tool_calls(tool_calls))

                if invalid_tool_calls:
                    print("Invalid Tool Calls:")
                    print(format_tool_calls(invalid_tool_calls))

                if response_metadata and response_metadata.get("finish_reason"):
                    print(f"Response Metadata: Finish Reason - {response_metadata['finish_reason']}")                    
        print("-" * 50)